# 04-03. Four-box Parameter Experiments  
# 4-box モデルでパラメタ感度実験をする

**Ocean Box Modeling with Python / 海洋ボックスモデル入門**

---

## 今日の目的 / Goals

`04-01` では、なぜ 4-box モデルが必要なのかを学びました。  
In `04-01`, we learned why a 4-box model is needed.

`04-02` では、4-box モデルの保存則とコードの構造を読みました。  
In `04-02`, we read the conservation laws and code structure of the 4-box model.

この Notebook では、いよいよ **モデルで実験** します。  
In this notebook, we finally **do experiments with the model**.

ここで大事なのは、ただ図を見ることではありません。  
The important thing is not just to look at plots.

毎回、次の順番で進めます。  
Each experiment follows this sequence:

```text
予想する
↓
実行する
↓
図を見る
↓
なぜそうなったか説明する
```

```text
predict
↓
run
↓
plot
↓
explain why
```

今日扱う実験は次です。  
Today we run the following experiments:

1. 北大西洋沈み込み循環 `Tcir`  
   North Atlantic sinking-like circulation `Tcir`

2. 南大洋深層ベンチレーション `FHD`  
   Southern Ocean deep ventilation `FHD`

3. 低緯度生物ポンプ `LF`  
   Low-latitude biological pump `LF`

4. 南大洋輸出生産 `CEPH`  
   Southern Ocean export production `CEPH`

5. 大気海洋 CO2 交換 `air_sea`  
   Air-sea CO2 exchange `air_sea`

6. どのパラメタが一番効くか  
   Which parameter matters most?

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import math

plt.rcParams["figure.figsize"] = (7, 4)

## 1. 4-box モデルの準備 / Prepare the 4-box model

この Notebook は単独で動くように、必要な関数をもう一度定義します。  
This notebook is self-contained, so we define the necessary functions again.

モデルの箱は次の 4 つです。  
The model has four boxes:

| Box | 日本語 | English |
|---|---|---|
| H | 南大洋表層 | Southern Ocean surface |
| L | 低緯度表層 | Low-latitude surface |
| N | 北大西洋表層 | North Atlantic surface |
| D | 深層 | Deep ocean |

主な循環は次のように考えます。  
The main circulation is represented as:

```text
D → H → L → N → D
```

H は南大洋の湧昇、N は北大西洋の沈み込みを表します。  
H represents Southern Ocean upwelling, and N represents North Atlantic sinking.

In [ ]:
# ============================================================
# Constants and helper functions
# ============================================================

CV1 = 1.0250e3    # mol/kg -> mol/m3
CV2 = 9.7561e-4   # mol/m3 -> mol/kg
CV3 = 1.0e6       # atm -> ppmv
CV4 = 3.1536e7    # seconds/year
CV5 = 8.64e4      # seconds/day

def to_umolkg(x):
    return CV2 * 1.0e6 * x

def to_ppmv(x):
    return CV3 * x

# Geometry
VOCN_TOTAL = 1.292e18
AOCN = 3.49e14
VATM = 1.773e20

FH_area = 0.10
FN_area = 0.15
FL_area = 0.75

ZH = 250.0
ZN = 250.0
ZL = 100.0

AOCNH = AOCN * FH_area
AOCNN = AOCN * FN_area
AOCNL = AOCN * FL_area

VOCNH = AOCNH * ZH
VOCNN = AOCNN * ZN
VOCNL = AOCNL * ZL
VOCND = VOCN_TOTAL - VOCNH - VOCNN - VOCNL

VOLUME = {"H": VOCNH, "L": VOCNL, "N": VOCNN, "D": VOCND}

TEMP = {"H": 1.0, "L": 21.5, "N": 3.0, "D": 1.75}
SALT = {"H": 34.7, "L": 34.7, "N": 34.7, "D": 34.7}

DT = 8.64e4

# Stoichiometry
RCP = 106.0
RNP = 16.0
RRC_DEFAULT = 0.20
RO2P = 172.0

# Biological export parameters
R = 1.0 / CV4
HSC = 2.0e-8 * CV1
DEL = 100.0

# Gas exchange coefficients
PVH = PVL = PVN = 3.0
FA = {
    "H": PVH * AOCNH / CV5,
    "L": PVL * AOCNL / CV5,
    "N": PVN * AOCNN / CV5,
}

In [ ]:
def o2sat(TEM, SAL):
    N1 = -1.734292e2
    N2 = +2.496339e2
    N3 = +1.433483e2
    N4 = -2.184920e1
    N5 = -3.309600e-2
    N6 = +1.425900e-2
    N7 = -1.700000e-3
    ATEM = TEM + 273.15
    O2S = math.exp(
        N1 + N2 * 1.0e2 / ATEM
        + N3 * math.log(ATEM / 1.0e2)
        + N4 * ATEM / 1.0e2
        + SAL * (N5 + N6 * ATEM / 1.0e2 + N7 * (ATEM / 1.0e2) ** 2)
    )
    return O2S * 4.35e1 * 1.025e-3

def chemeq_const(TEM, SAL):
    ATEM = TEM + 273.15
    TK = ATEM * 1.0e-2
    SK = 2.3517e-2 + (-2.3656e-2 + 4.7036e-3 * TK) * TK
    K0 = math.exp(-6.02409e1 + 9.34517e1 / TK + 2.33585e1 * math.log(TK) + SK * SAL)
    K1 = math.exp(math.log(10.0) * (
        13.7201 - 3.1334e-2 * ATEM - 3.23576e3 / ATEM
        - 1.3e-5 * SAL * ATEM + 1.032e-1 * math.sqrt(SAL)
    ))
    safe_sal = SAL if SAL > 0 else 1.0
    K2 = math.exp(math.log(10.0) * (
        -5.3719645e3 - 1.671221e0 * ATEM - 2.2913e-1 * SAL
        - 1.83802e1 * math.log10(safe_sal) + 1.2837528e5 / ATEM
        + 2.1943005e3 * math.log10(ATEM) + 8.0944e-4 * SAL * ATEM
        + 5.61711e3 * math.log10(safe_sal) / ATEM - 2.136e0 * SAL / ATEM
    ))
    KB = math.exp(math.log(10.0) * (-9.26 + 8.86e-3 * SAL + 1e-3 * TEM))
    KW = math.exp(
        +1.489802e2 - 1.384726e4 / ATEM - 2.36521e1 * math.log(ATEM)
        + ((-7.92447e1 + 3.29872e3 / ATEM + 1.20408e1 * math.log(ATEM))
           * math.sqrt(SAL) - 1.9813e-2 * SAL)
    )
    BT = 4.106e-4 * SAL / 35.0
    return BT, K0, K1, K2, KB, KW

def co2_nibun(BT, K0, K1, K2, KB, KW, AT, CT):
    ATX = CV2 * AT
    CTX = CV2 * CT
    h_low, h_high = 1.0e-14, 1.0
    hx = 0.5 * (h_low + h_high)

    for _ in range(100000):
        h = 0.5 * (h_low + h_high)
        denom = h*h + K1*h + K1*K2
        at_calc = (
            (2.0*K1*K2*CTX)/denom
            + (h*K1*CTX)/denom
            + (KB*BT)/(h+KB)
            + KW/h
            - h
        )
        if abs(ATX - at_calc) <= 1.0e-15:
            hx = h
            break
        if at_calc < ATX:
            h_high = h
        else:
            h_low = h

    denom2 = hx*hx + K1*hx + K1*K2
    PCO2 = (hx*hx*CTX) / denom2 / K0
    CO2 = (hx*hx*CTX) / denom2 / CV2
    HCO3 = K1 * CO2 / hx
    CO32 = K2 * HCO3 / hx
    return CO2, HCO3, CO32, PCO2

def carbonate_box(temp, sal, alk, dic):
    BT, K0, K1, K2, KB, KW = chemeq_const(temp, sal)
    CO2, HCO3, CO32, PCO2 = co2_nibun(BT, K0, K1, K2, KB, KW, alk, dic)
    return {"K0": K0, "CO2": CO2, "HCO3": HCO3, "CO32": CO32, "PCO2": PCO2}

In [ ]:
def initial_four_box():
    x = {}
    for b in ["H", "L", "N", "D"]:
        x[f"PO4{b}"] = 2.10e-6 * CV1
        x[f"DIC{b}"] = 2.235e-3 * CV1
        x[f"ALK{b}"] = 2.374e-3 * CV1
        x[f"DO2{b}"] = 1.60e-4 * CV1
    x["PCO2A"] = 280.0 / CV3
    return x

def compute_exports(x, CEPH=0.75, CEPN=0.02, LF=0.5):
    EPH = (CEPH / RCP) * AOCNH / CV4
    EPN = (CEPN / RCP) * AOCNN / CV4
    EPL = R * DEL * LF * x["PO4L"] * (x["PO4L"] / (HSC + x["PO4L"])) * VOCNL
    return {"H": EPH, "L": EPL, "N": EPN}

def one_step_four_box(
    x,
    Tcir=2.0e7,
    FHD=6.0e7,
    LF=0.5,
    CEPH=0.75,
    CEPN=0.02,
    RRC=RRC_DEFAULT,
    air_sea=True,
):
    y = dict(x)

    exports = compute_exports(x, CEPH=CEPH, CEPN=CEPN, LF=LF)
    EPH, EPL, EPN = exports["H"], exports["L"], exports["N"]

    alk_factor = 2.0 * RRC * RCP - RNP
    dic_factor = (1.0 + RRC) * RCP

    gas = {"H": 0.0, "L": 0.0, "N": 0.0}
    if air_sea:
        for b in ["H", "L", "N"]:
            c = carbonate_box(TEMP[b], SALT[b], x[f"ALK{b}"], x[f"DIC{b}"])
            gas[b] = FA[b] * CV1 * c["K0"] * (x["PCO2A"] - c["PCO2"])

    def update_tracer(prefix, bio_factor, gas_terms=None):
        if gas_terms is None:
            gas_terms = {"H": 0.0, "L": 0.0, "N": 0.0}

        y[f"{prefix}H"] = x[f"{prefix}H"] + (
            Tcir * (x[f"{prefix}D"] - x[f"{prefix}H"])
            + FHD * (x[f"{prefix}D"] - x[f"{prefix}H"])
            - bio_factor * EPH
            + gas_terms.get("H", 0.0)
        ) * (DT / VOLUME["H"])

        y[f"{prefix}L"] = x[f"{prefix}L"] + (
            Tcir * (x[f"{prefix}H"] - x[f"{prefix}L"])
            - bio_factor * EPL
            + gas_terms.get("L", 0.0)
        ) * (DT / VOLUME["L"])

        y[f"{prefix}N"] = x[f"{prefix}N"] + (
            Tcir * (x[f"{prefix}L"] - x[f"{prefix}N"])
            - bio_factor * EPN
            + gas_terms.get("N", 0.0)
        ) * (DT / VOLUME["N"])

        y[f"{prefix}D"] = x[f"{prefix}D"] + (
            Tcir * (x[f"{prefix}N"] - x[f"{prefix}D"])
            + FHD * (x[f"{prefix}H"] - x[f"{prefix}D"])
            + bio_factor * (EPH + EPL + EPN)
        ) * (DT / VOLUME["D"])

    update_tracer("PO4", 1.0)
    update_tracer("ALK", alk_factor)
    update_tracer("DIC", dic_factor, gas_terms=gas)

    O2SAT = {b: o2sat(TEMP[b], SALT[b]) for b in ["H", "L", "N", "D"]}

    y["DO2H"] = x["DO2H"] + (
        Tcir * (O2SAT["D"] - x["DO2H"])
        + FHD * (O2SAT["H"] - x["DO2H"])
        + RO2P * EPH
        + FA["H"] * (O2SAT["H"] - x["DO2H"])
    ) * (DT / VOLUME["H"])

    y["DO2L"] = x["DO2L"] + (
        Tcir * (x["DO2H"] - x["DO2L"])
        + RO2P * EPL
        + FA["L"] * (O2SAT["L"] - x["DO2L"])
    ) * (DT / VOLUME["L"])

    y["DO2N"] = x["DO2N"] + (
        Tcir * (x["DO2L"] - x["DO2N"])
        + RO2P * EPN
        + FA["N"] * (O2SAT["N"] - x["DO2N"])
    ) * (DT / VOLUME["N"])

    y["DO2D"] = x["DO2D"] + (
        Tcir * (x["DO2N"] - x["DO2D"])
        + FHD * (x["DO2H"] - x["DO2D"])
        - RO2P * (EPH + EPL + EPN)
    ) * (DT / VOLUME["D"])

    if air_sea:
        flux_to_atm = 0.0
        for b in ["H", "L", "N"]:
            cnew = carbonate_box(TEMP[b], SALT[b], y[f"ALK{b}"], y[f"DIC{b}"])
            flux_to_atm += FA[b] * CV1 * cnew["K0"] * (cnew["PCO2"] - x["PCO2A"])
        y["PCO2A"] = x["PCO2A"] + flux_to_atm * (DT / VATM)

    diagnostics = {"EPH": EPH, "EPL": EPL, "EPN": EPN}
    return y, diagnostics

def diagnose_four_box(x, diag=None):
    row = {}
    for b in ["H", "L", "N", "D"]:
        c = carbonate_box(TEMP[b], SALT[b], x[f"ALK{b}"], x[f"DIC{b}"])
        row[f"PO4{b}"] = to_umolkg(x[f"PO4{b}"])
        row[f"DIC{b}"] = to_umolkg(x[f"DIC{b}"])
        row[f"ALK{b}"] = to_umolkg(x[f"ALK{b}"])
        row[f"DO2{b}"] = to_umolkg(x[f"DO2{b}"])
        row[f"PCO2{b}"] = to_ppmv(c["PCO2"])
        row[f"CO3{b}"] = to_umolkg(c["CO32"])
    row["PCO2A"] = to_ppmv(x["PCO2A"])
    if diag is not None:
        row["EPH_PgCyr"] = diag["H"] * RCP * 12.0e-15 * CV4
        row["EPL_PgCyr"] = diag["L"] * RCP * 12.0e-15 * CV4
        row["EPN_PgCyr"] = diag["N"] * RCP * 12.0e-15 * CV4
    return row

def run_four_box(
    years=3000,
    Tcir=2.0e7,
    FHD=6.0e7,
    LF=0.5,
    CEPH=0.75,
    CEPN=0.02,
    RRC=RRC_DEFAULT,
    air_sea=True,
):
    x = initial_four_box()
    history = []

    for day in range(years * 365 + 1):
        if day % 365 == 0:
            _, diag = one_step_four_box(
                x, Tcir=Tcir, FHD=FHD, LF=LF,
                CEPH=CEPH, CEPN=CEPN, RRC=RRC, air_sea=air_sea
            )
            row = {"year": day / 365}
            row.update(diagnose_four_box(x, diag=diag))
            history.append(row)

        x, diag = one_step_four_box(
            x, Tcir=Tcir, FHD=FHD, LF=LF,
            CEPH=CEPH, CEPN=CEPN, RRC=RRC, air_sea=air_sea
        )

    return x, pd.DataFrame(history)

## 2. 標準実験 / Standard experiment

まず標準実験を走らせます。  
First, we run the standard experiment.

この標準実験を基準にして、あとでパラメタを変えます。  
We use this standard experiment as a baseline for later parameter changes.

In [ ]:
baseline_final, baseline = run_four_box(years=3000)

baseline.tail()

In [ ]:
def plot_time_series(hist, columns, ylabel, title):
    plt.figure()
    for col in columns:
        plt.plot(hist["year"], hist[col], label=col)
    plt.xlabel("Time [yr]")
    plt.ylabel(ylabel)
    plt.title(title)
    plt.legend()
    plt.grid(True)
    plt.show()

plot_time_series(baseline, ["PCO2A", "PCO2H", "PCO2L", "PCO2N"], "pCO2 [ppmv]", "Standard experiment: pCO2")
plot_time_series(baseline, ["PO4H", "PO4L", "PO4N", "PO4D"], "PO4 [umol/kg]", "Standard experiment: PO4")
plot_time_series(baseline, ["DO2H", "DO2L", "DO2N", "DO2D"], "O2 [umol/kg]", "Standard experiment: O2")

## 3. Experiment 1: 北大西洋沈み込み循環 `Tcir`  
## Experiment 1: North Atlantic sinking-like circulation `Tcir`

`Tcir` は、次の循環の強さを表します。  
`Tcir` represents the strength of this circulation:

```text
D → H → L → N → D
```

この循環を強くすると、表層と深層のつながりが強くなります。  
If this circulation becomes stronger, the connection between surface and deep ocean becomes stronger.

この循環を弱くすると、北大西洋沈み込みが弱くなるような実験になります。  
If it becomes weaker, it resembles a weakened North Atlantic sinking experiment.

### 予想 / Prediction

`Tcir` を小さくすると、N box の PO4 と深層 O2 はどう変わるでしょうか。  
If `Tcir` is reduced, what happens to PO4 in the N box and O2 in the deep box?

In [ ]:
Tcir_list = [0.5e7, 1.0e7, 2.0e7, 4.0e7]
summary_T = []

plt.figure()
for t in Tcir_list:
    _, h = run_four_box(years=3000, Tcir=t)
    plt.plot(h["year"], h["PCO2A"], label=f"Tcir={t:.1e}")
    summary_T.append({
        "Tcir": t,
        "Final pCO2A": h["PCO2A"].iloc[-1],
        "Final PO4N": h["PO4N"].iloc[-1],
        "Final DO2D": h["DO2D"].iloc[-1],
        "Final DICD": h["DICD"].iloc[-1],
    })

plt.xlabel("Time [yr]")
plt.ylabel("Atmospheric pCO2 [ppmv]")
plt.title("Experiment 1: sensitivity to Tcir")
plt.legend()
plt.grid(True)
plt.show()

pd.DataFrame(summary_T)

### 考察 / Interpretation

`Tcir` を変えると、N box と D box のつながりが変わります。  
Changing `Tcir` changes the connection between the N box and the D box.

大きな `Tcir` は、深層水形成・大規模循環が強い状態を表します。  
A large `Tcir` represents strong deep-water formation and large-scale circulation.

小さな `Tcir` は、深層がより孤立する状態に近づきます。  
A small `Tcir` makes the deep ocean more isolated.

**問い / Question**

この実験で、大気 pCO2 の変化は大きいですか。  
Is the atmospheric pCO2 response large in this experiment?

もし小さい場合、それはなぜでしょうか。  
If it is small, why might that be?

## 4. Experiment 2: 南大洋深層ベンチレーション `FHD`  
## Experiment 2: Southern Ocean deep ventilation `FHD`

`FHD` は、H box と D box の直接交換を表します。  
`FHD` represents direct exchange between the H box and the D box.

H box は南大洋表層を表すので、これは南大洋の深層ベンチレーションに近い意味を持ちます。  
Since H represents the Southern Ocean surface, this parameter is related to Southern Ocean deep ventilation.

Toggweiler 型の議論では、深層水のベンチレーションを弱めると、大気 CO2 が下がる可能性があります。  
In Toggweiler-type arguments, weaker deep-water ventilation can lower atmospheric CO2.

### 予想 / Prediction

`FHD` を小さくすると、大気 pCO2、深層 DIC、深層 O2 はどう変わるでしょうか。  
If `FHD` is reduced, what happens to atmospheric pCO2, deep DIC, and deep O2?

In [ ]:
FHD_list = [1.5e7, 3.0e7, 6.0e7, 9.0e7, 1.2e8]
summary_FHD = []

plt.figure()
for fhd in FHD_list:
    _, h = run_four_box(years=3000, FHD=fhd)
    plt.plot(h["year"], h["PCO2A"], label=f"FHD={fhd:.1e}")
    summary_FHD.append({
        "FHD": fhd,
        "Final pCO2A": h["PCO2A"].iloc[-1],
        "Final DICD": h["DICD"].iloc[-1],
        "Final DO2D": h["DO2D"].iloc[-1],
        "Final PO4H": h["PO4H"].iloc[-1],
    })

plt.xlabel("Time [yr]")
plt.ylabel("Atmospheric pCO2 [ppmv]")
plt.title("Experiment 2: sensitivity to FHD")
plt.legend()
plt.grid(True)
plt.show()

pd.DataFrame(summary_FHD)

In [ ]:
df_FHD = pd.DataFrame(summary_FHD)

plt.figure()
plt.plot(df_FHD["FHD"], df_FHD["Final pCO2A"], marker="o")
plt.xlabel("FHD [m3/s]")
plt.ylabel("Final atmospheric pCO2 [ppmv]")
plt.title("Final pCO2A vs FHD")
plt.grid(True)
plt.show()

plt.figure()
plt.plot(df_FHD["FHD"], df_FHD["Final DO2D"], marker="o")
plt.xlabel("FHD [m3/s]")
plt.ylabel("Final deep O2 [umol/kg]")
plt.title("Final deep O2 vs FHD")
plt.grid(True)
plt.show()

### 考察 / Interpretation

`FHD` は大気 pCO2 に効きやすいパラメタです。  
`FHD` is often an effective parameter for atmospheric pCO2.

なぜなら、H box は大気と接し、さらに深層 D と直接つながっているからです。  
This is because the H box is in contact with the atmosphere and also directly connected to the deep box D.

**問い / Question**

`FHD` を小さくしたとき、深層 O2 はどうなりましたか。  
What happened to deep O2 when `FHD` was reduced?

これは「深層がベンチレートされにくくなる」ことと整合的ですか。  
Is this consistent with weaker deep-ocean ventilation?

## 5. Experiment 3: 低緯度生物ポンプ `LF`  
## Experiment 3: Low-latitude biological pump `LF`

`LF` は、L box の輸出生産 `EPL` の強さに関係します。  
`LF` controls the strength of export production `EPL` in the L box.

L box は面積が大きいため、低緯度生物ポンプは全体の炭素循環に効く可能性があります。  
Because the L box has a large area, the low-latitude biological pump can affect the whole carbon cycle.

### 予想 / Prediction

`LF` を大きくすると、表層 DIC、大気 pCO2、深層 DIC はどう変わるでしょうか。  
If `LF` is increased, what happens to surface DIC, atmospheric pCO2, and deep DIC?

In [ ]:
LF_list = [0.2, 0.5, 1.0, 2.0]
summary_LF = []

plt.figure()
for lf in LF_list:
    _, h = run_four_box(years=3000, LF=lf)
    plt.plot(h["year"], h["PCO2A"], label=f"LF={lf}")
    summary_LF.append({
        "LF": lf,
        "Final pCO2A": h["PCO2A"].iloc[-1],
        "Final DICL": h["DICL"].iloc[-1],
        "Final DICD": h["DICD"].iloc[-1],
        "Final EPL": h["EPL_PgCyr"].iloc[-1],
        "Final PO4L": h["PO4L"].iloc[-1],
    })

plt.xlabel("Time [yr]")
plt.ylabel("Atmospheric pCO2 [ppmv]")
plt.title("Experiment 3: sensitivity to LF")
plt.legend()
plt.grid(True)
plt.show()

pd.DataFrame(summary_LF)

In [ ]:
df_LF = pd.DataFrame(summary_LF)

plt.figure()
plt.plot(df_LF["LF"], df_LF["Final EPL"], marker="o")
plt.xlabel("LF")
plt.ylabel("Final EPL [PgC/yr]")
plt.title("Low-latitude export vs LF")
plt.grid(True)
plt.show()

plt.figure()
plt.plot(df_LF["LF"], df_LF["Final pCO2A"], marker="o")
plt.xlabel("LF")
plt.ylabel("Final atmospheric pCO2 [ppmv]")
plt.title("Atmospheric pCO2 vs LF")
plt.grid(True)
plt.show()

### 考察 / Interpretation

生物ポンプを強くすると、表層から DIC と PO4 が取り除かれ、深層へ送られます。  
A stronger biological pump removes DIC and PO4 from the surface and transfers them to the deep ocean.

そのため、表層 pCO2 は下がりやすく、大気 pCO2 も下がる方向に動きます。  
Therefore, surface pCO2 tends to decrease, and atmospheric pCO2 tends to decrease.

ただし、応答は炭酸系と ALK の変化にも依存します。  
However, the response also depends on carbonate chemistry and changes in alkalinity.

## 6. Experiment 4: 南大洋輸出生産 `CEPH`  
## Experiment 4: Southern Ocean export `CEPH`

`CEPH` は H box、つまり南大洋表層の輸出生産を表します。  
`CEPH` represents export production in the H box, the Southern Ocean surface.

南大洋は深層水と大気の接点なので、ここでの生物ポンプは大気 CO2 に強く効く可能性があります。  
The Southern Ocean is a contact point between deep water and the atmosphere, so biological export there can strongly affect atmospheric CO2.

### 予想 / Prediction

`CEPH` を大きくすると、H box の PO4 と大気 pCO2 はどう変わるでしょうか。  
If `CEPH` is increased, what happens to H-box PO4 and atmospheric pCO2?

In [ ]:
CEPH_list = [0.0, 0.25, 0.75, 1.5, 3.0]
summary_CEPH = []

plt.figure()
for ceph in CEPH_list:
    _, h = run_four_box(years=3000, CEPH=ceph)
    plt.plot(h["year"], h["PCO2A"], label=f"CEPH={ceph}")
    summary_CEPH.append({
        "CEPH": ceph,
        "Final pCO2A": h["PCO2A"].iloc[-1],
        "Final PO4H": h["PO4H"].iloc[-1],
        "Final DICH": h["DICH"].iloc[-1],
        "Final EPH": h["EPH_PgCyr"].iloc[-1],
    })

plt.xlabel("Time [yr]")
plt.ylabel("Atmospheric pCO2 [ppmv]")
plt.title("Experiment 4: sensitivity to CEPH")
plt.legend()
plt.grid(True)
plt.show()

pd.DataFrame(summary_CEPH)

### 考察 / Interpretation

`CEPH` を大きくすると、H box から DIC と PO4 が深層へ輸送されやすくなります。  
Increasing `CEPH` makes it easier to transfer DIC and PO4 from H to the deep ocean.

H box は大気と接しているので、この変化は大気 pCO2 に影響します。  
Because H is in contact with the atmosphere, this affects atmospheric pCO2.

**問い / Question**

同じ生物ポンプでも、`LF` と `CEPH` の効果は同じでしたか。  
Were the effects of `LF` and `CEPH` the same?

違うとしたら、H と L の役割の違いから説明してください。  
If different, explain using the difference between the roles of H and L.

## 7. Experiment 5: CaCO3 比 `RRC`  
## Experiment 5: CaCO3 rain ratio `RRC`

`RRC` は有機炭素輸出に対する CaCO3 輸出の比を表します。  
`RRC` represents the ratio of CaCO3 export to organic carbon export.

この値を変えると、DIC だけでなく ALK も変わります。  
Changing this value affects not only DIC but also ALK.

炭酸系では、ALK の変化が pCO2 に大きく効きます。  
In carbonate chemistry, changes in ALK strongly affect pCO2.

### 予想 / Prediction

`RRC` を大きくすると、大気 pCO2 は上がるでしょうか、下がるでしょうか。  
If `RRC` is increased, does atmospheric pCO2 increase or decrease?

In [ ]:
RRC_list = [0.0, 0.1, 0.2, 0.3, 0.4]
summary_RRC = []

plt.figure()
for rrc in RRC_list:
    _, h = run_four_box(years=3000, RRC=rrc)
    plt.plot(h["year"], h["PCO2A"], label=f"RRC={rrc}")
    summary_RRC.append({
        "RRC": rrc,
        "Final pCO2A": h["PCO2A"].iloc[-1],
        "Final ALKL": h["ALKL"].iloc[-1],
        "Final DICL": h["DICL"].iloc[-1],
        "Final CO3L": h["CO3L"].iloc[-1],
    })

plt.xlabel("Time [yr]")
plt.ylabel("Atmospheric pCO2 [ppmv]")
plt.title("Experiment 5: sensitivity to RRC")
plt.legend()
plt.grid(True)
plt.show()

pd.DataFrame(summary_RRC)

### 考察 / Interpretation

CaCO3 の輸出は DIC と ALK の両方を変えます。  
CaCO3 export changes both DIC and ALK.

有機物ポンプだけでなく、炭酸カルシウムポンプも大気 CO2 に効きます。  
Not only the organic carbon pump, but also the calcium carbonate pump affects atmospheric CO2.

**問い / Question**

`RRC` を変えたとき、`ALKL`, `DICL`, `CO3L` はどう変わりましたか。  
How did `ALKL`, `DICL`, and `CO3L` change when `RRC` was varied?

## 8. Experiment 6: 大気海洋 CO2 交換の ON/OFF  
## Experiment 6: Air-sea CO2 exchange ON/OFF

`air_sea=False` にすると、大気海洋 CO2 交換を止めます。  
If `air_sea=False`, air-sea CO2 exchange is turned off.

この場合、大気 pCO2 は変化しません。  
In this case, atmospheric pCO2 does not change.

しかし、海洋内部では輸送と生物ポンプが動いているので、PO4, DIC, O2 は変化します。  
However, inside the ocean, transport and biological pump still operate, so PO4, DIC, and O2 change.

In [ ]:
_, h_air = run_four_box(years=3000, air_sea=True)
_, h_noair = run_four_box(years=3000, air_sea=False)

plt.figure()
plt.plot(h_air["year"], h_air["PCO2A"], label="air_sea=True")
plt.plot(h_noair["year"], h_noair["PCO2A"], label="air_sea=False")
plt.xlabel("Time [yr]")
plt.ylabel("Atmospheric pCO2 [ppmv]")
plt.title("Air-sea CO2 exchange ON/OFF")
plt.legend()
plt.grid(True)
plt.show()

plt.figure()
plt.plot(h_air["year"], h_air["DICD"], label="DICD air_sea=True")
plt.plot(h_noair["year"], h_noair["DICD"], label="DICD air_sea=False")
plt.xlabel("Time [yr]")
plt.ylabel("Deep DIC [umol/kg]")
plt.title("Deep DIC with/without air-sea exchange")
plt.legend()
plt.grid(True)
plt.show()

## 9. どのパラメタが一番効くか / Which parameter matters most?

ここまでいろいろなパラメタを変えました。  
So far we changed many parameters.

最後に、標準実験からの大気 pCO2 変化量を比較します。  
Finally, we compare the change in atmospheric pCO2 relative to the standard experiment.

これは、簡単な「感度ランキング」です。  
This is a simple "sensitivity ranking."

In [ ]:
baseline_pco2 = baseline["PCO2A"].iloc[-1]

experiments = []

cases = [
    ("Tcir low", {"Tcir": 0.5e7}),
    ("Tcir high", {"Tcir": 4.0e7}),
    ("FHD low", {"FHD": 1.5e7}),
    ("FHD high", {"FHD": 1.2e8}),
    ("LF low", {"LF": 0.2}),
    ("LF high", {"LF": 2.0}),
    ("CEPH low", {"CEPH": 0.0}),
    ("CEPH high", {"CEPH": 3.0}),
    ("RRC low", {"RRC": 0.0}),
    ("RRC high", {"RRC": 0.4}),
]

for name, kwargs in cases:
    _, h = run_four_box(years=3000, **kwargs)
    final_pco2 = h["PCO2A"].iloc[-1]
    experiments.append({
        "Case": name,
        "Final pCO2A": final_pco2,
        "Delta pCO2A from baseline": final_pco2 - baseline_pco2,
        "Absolute change": abs(final_pco2 - baseline_pco2),
    })

df_sens = pd.DataFrame(experiments).sort_values("Absolute change", ascending=False)
df_sens

In [ ]:
plt.figure(figsize=(8, 5))
plt.barh(df_sens["Case"], df_sens["Delta pCO2A from baseline"])
plt.xlabel("Delta atmospheric pCO2 from baseline [ppmv]")
plt.title("Parameter sensitivity ranking")
plt.grid(True, axis="x")
plt.show()

### 考察 / Interpretation

この表と図から、どのパラメタが大気 pCO2 に効きやすいかが分かります。  
This table and figure show which parameters most strongly affect atmospheric pCO2.

ただし、これはこの単純な 4-box モデルの範囲での結果です。  
However, this result is only within this simplified 4-box model.

実際の海洋では、空間構造、海氷、風、混合、栄養塩制限なども重要です。  
In the real ocean, spatial structure, sea ice, winds, mixing, and nutrient limitation are also important.

## 10. Research mode: LGM の低 CO2 を作るなら？  
## 10. Research mode: How would you make low LGM CO2?

最後に、研究者モードの課題です。  
Finally, a research-mode exercise.

あなたは第四紀の気候研究者だとします。  
Suppose you are a Quaternary climate researcher.

目標は、氷期の低い大気 CO2 を単純な 4-box モデルで再現することです。  
Your goal is to reproduce low glacial atmospheric CO2 using a simple 4-box model.

### 問い / Question

大気 pCO2 を下げるために、どのパラメタを変えますか。  
Which parameters would you change to lower atmospheric pCO2?

候補:  
Candidates:

```text
FHD  : Southern Ocean deep ventilation
Tcir : large-scale circulation
LF   : low-latitude biological pump
CEPH : Southern Ocean export
RRC  : CaCO3 rain ratio
```

### 書くべきこと / What to write

1. どのパラメタを変えるか。  
   Which parameter you change.

2. どちら向きに変えるか。  
   In which direction you change it.

3. なぜ大気 CO2 が下がると考えるか。  
   Why you think atmospheric CO2 decreases.

4. その副作用として、O2, DIC, PO4 はどう変わるか。  
   As a side effect, how O2, DIC, and PO4 change.

In [ ]:
# Try your own LGM-like experiment.
# 自分の LGM 風実験を試してください。

my_Tcir = 2.0e7
my_FHD = 1.5e7
my_LF = 1.0
my_CEPH = 1.5
my_RRC = 0.2

final_lgm, hist_lgm = run_four_box(
    years=3000,
    Tcir=my_Tcir,
    FHD=my_FHD,
    LF=my_LF,
    CEPH=my_CEPH,
    RRC=my_RRC,
)

plt.figure()
plt.plot(baseline["year"], baseline["PCO2A"], label="baseline")
plt.plot(hist_lgm["year"], hist_lgm["PCO2A"], label="my LGM-like experiment")
plt.xlabel("Time [yr]")
plt.ylabel("Atmospheric pCO2 [ppmv]")
plt.title("Your LGM-like 4-box experiment")
plt.legend()
plt.grid(True)
plt.show()

pd.DataFrame({
    "Case": ["baseline", "my experiment"],
    "Final pCO2A": [baseline["PCO2A"].iloc[-1], hist_lgm["PCO2A"].iloc[-1]],
    "Final DICD": [baseline["DICD"].iloc[-1], hist_lgm["DICD"].iloc[-1]],
    "Final DO2D": [baseline["DO2D"].iloc[-1], hist_lgm["DO2D"].iloc[-1]],
    "Final PO4H": [baseline["PO4H"].iloc[-1], hist_lgm["PO4H"].iloc[-1]],
})

## 11. 課題 / Exercises

### 課題 1 / Exercise 1

`Tcir` を小さくしたとき、N box の PO4 と深層 O2 はどう変化したか。  
When `Tcir` was reduced, how did PO4 in the N box and deep O2 change?

### 課題 2 / Exercise 2

`FHD` を小さくしたとき、大気 pCO2、深層 DIC、深層 O2 はどう変化したか。  
When `FHD` was reduced, how did atmospheric pCO2, deep DIC, and deep O2 change?

### 課題 3 / Exercise 3

`LF` と `CEPH` の効果は同じか。違う場合、なぜ違うのか。  
Are the effects of `LF` and `CEPH` the same? If not, why are they different?

### 課題 4 / Exercise 4

`RRC` を変えると、なぜ DIC だけでなく ALK や CO3 も変わるのか。  
Why do ALK and CO3, not only DIC, change when `RRC` is varied?

### 課題 5 / Exercise 5

大気 pCO2 を下げるために最も有効そうなパラメタを 1 つ選び、理由を説明せよ。  
Choose one parameter that seems most effective for lowering atmospheric pCO2 and explain why.

## 12. 次へ / Next step

この Notebook では、4-box モデルを使って感度実験を行いました。  
In this notebook, we performed sensitivity experiments with the 4-box model.

次は、モデルを自分で改造する段階に進みます。  
Next, we move to modifying the model ourselves.

```text
04-04:
  Add your own experiment
  Modify model equations
  Compare scenarios
  Prepare a short research report
```

つまり、モデルを「使う」段階から、モデルを「作り変える」段階へ進みます。  
We move from using the model to modifying the model.